# Project Introduction — Bank Customer Churn Prediction with an Artificial Neural Network

### 📌 What this project is

An **end-to-end deep learning project**: build an **Artificial Neural Network (ANN)** with **Keras/TensorFlow** to solve a **binary classification** problem on tabular data, then deploy it as a web app.

**Problem statement:** given a bank customer's profile, predict whether they will **leave the bank ("churn")** or not.

This notebook is a planning/setup overview — no model training happens here yet. It documents the problem, the dataset, the full pipeline, and the local environment setup, matching the project-intro video.

## 1. The dataset — `Churn_Modelling.csv`

Already present in the project folder (one level up from this `notebooks/` folder). Each row is a bank customer; the target column is **`Exited`** (`1` = left the bank, `0` = stayed).

In [1]:
import pandas as pd

df = pd.read_csv("../Churn_Modelling.csv")

print("Shape:", df.shape)
df.head()

Shape: (10000, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.1 MB


**Columns, by role:**

| Role | Columns |
|---|---|
| **Identifiers (not predictive — drop before modeling)** | `RowNumber`, `CustomerId`, `Surname` |
| **Features** | `CreditScore`, `Geography`, `Gender`, `Age`, `Tenure`, `Balance`, `NumOfProducts`, `HasCrCard`, `IsActiveMember`, `EstimatedSalary` |
| **Target** | `Exited` |

`Geography` (`France`/`Spain`/`Germany`) and `Gender` (`Female`/`Male`) are categorical and will need encoding before they can be fed to a neural network.

In [3]:
print("Geography categories:", df["Geography"].unique().tolist())
print("Gender categories   :", df["Gender"].unique().tolist())
print("\nMissing values per column (none expected in this dataset):")
print(df.isnull().sum().sum(), "total missing values")

print("\nTarget balance:")
print(df["Exited"].value_counts())
print(f"Churn rate: {df['Exited'].mean():.1%}")

Geography categories: ['France', 'Spain', 'Germany']
Gender categories   : ['Female', 'Male']

Missing values per column (none expected in this dataset):
0 total missing values

Target balance:
Exited
0    7963
1    2037
Name: count, dtype: int64
Churn rate: 20.4%


> 📝 **Worth noting for later (not mentioned in the video, but relevant when training):** the target is **imbalanced** — only about **20.4%** of customers churned (`Exited=1`). A model that just always predicts "stayed" would already be ~80% "accurate" without learning anything useful — so accuracy alone won't be a reliable metric later; precision/recall/F1 or a confusion matrix will matter more once training starts.
>
> **Why 11 input features, matching the video:** after dropping the 3 identifier columns, there are 10 raw feature columns. `Gender` (2 categories) becomes 1 encoded column, and `Geography` (3 categories: France/Spain/Germany) becomes 2 one-hot columns (dropping one category to avoid redundancy) — so `10 - 2 (raw categorical) + 1 (Gender) + 2 (Geography) = 11` final input features. This lines up exactly with the video's "11 nodes in the input layer.\"

## 2. Tech stack

| Library | Role |
|---|---|
| **TensorFlow** | Open-source deep learning framework — covers ANN, RNN, LSTM, GRU, Transformers, etc. |
| **Keras** | High-level API for building neural networks with much less code. Since TF 2.0, `tf.keras` is the integrated, recommended way to use Keras — no separate `keras` package install needed. (Separately, **Keras 3.0** also now supports JAX and PyTorch as backends, not just TensorFlow — useful to know, though this course sticks with TensorFlow throughout.) |
| **scikit-learn** | Feature engineering — encoding categorical variables, `StandardScaler` for standardization. |
| **pandas / numpy** | Data loading and manipulation. |
| **TensorBoard** | Visualizing training metrics (loss/accuracy curves) during and after training. |
| **matplotlib** | Plotting. |
| **Streamlit** | Building the web app that wraps the trained model, deployed to Streamlit Cloud. |

> 💡 **Why TensorFlow (and not also PyTorch)?** Both are open-source and widely used; the video's advice is to become proficient in **one**, since the underlying deep learning concepts transfer between them — including later, when working with LLMs/Generative AI, where switching frameworks is mostly a matter of translating the same ideas into a different API.

## 3. End-to-end project pipeline

```
 Churn_Modelling.csv
         │
         ▼
 Feature Engineering
   - drop identifier columns (RowNumber, CustomerId, Surname)
   - encode categorical variables (Geography, Gender) -> numeric
   - standardize numeric features (StandardScaler)
         │
         ▼
 Build the ANN (Keras/TensorFlow)
   Input layer (11 nodes)  ->  Hidden layer(s) + Dropout  ->  Output layer (1 node, sigmoid)
         │
         ▼
 Train
   forward propagation -> loss function -> optimizer -> backward propagation
   (Dropout randomly disables some nodes/weight-updates during training, to reduce overfitting)
         │
         ▼
 Save the trained model
   weights/architecture -> pickle file and/or .h5 file, for reuse without retraining
         │
         ▼
 Build a Streamlit web app around the saved model
         │
         ▼
 Deploy to Streamlit Cloud
```

Each stage will get its own notebook/step as the project progresses — this notebook is just the roadmap.

## 4. ANN architecture (preview)

- **Input layer:** 11 nodes — one per engineered feature (see Section 1).
- **Hidden layer(s):** at least one; the exact count and size is a design choice made when the model is actually built.
- **Dropout:** applied between layers to randomly "switch off" a fraction of nodes/weight-updates during training — a regularization technique to reduce overfitting.
- **Output layer:** 1 node — since this is **binary classification** (`Exited` is 0 or 1), the output layer will use a **sigmoid** activation to produce a probability between 0 and 1 (not explicitly named in the video, but implied by the binary target and standard practice for this kind of problem).

Full architecture, activation functions, loss function, and optimizer choices will be covered in the notebook where the model is actually built.

## 5. Local environment setup

The video sets this up locally with `conda`. One correction below: the stated TensorFlow version `2.1.50` isn't a real released version — the intended version is almost certainly **`2.15.0`** (a real TensorFlow release), which is what's used below.

```bash
# create a dedicated conda environment for this project
conda create -p venv python=3.11
conda activate venv/

# requirements.txt
tensorflow==2.15.0
pandas
numpy
scikit-learn
tensorboard
matplotlib
streamlit

# install everything
pip install -r requirements.txt
```

> 💡 **CPU vs. GPU:** the plain `tensorflow` pip package (2.1+) supports both CPU and GPU out of the box — no separate `tensorflow-gpu` package is needed anymore; GPU acceleration is used automatically if compatible drivers/CUDA are detected, otherwise it falls back to CPU. For this dataset's size (10,000 rows), CPU training will be fast enough.

## 6. Environment check (this notebook's current kernel)

Purely informational — this checks what's available in the kernel *this notebook happens to be running in* right now, not the dedicated project `conda` environment described above. It's fine (and expected) for TensorFlow/Streamlit to be missing here until that dedicated environment is created and used to run the later notebooks.

In [4]:
import importlib

required = ["tensorflow", "keras", "sklearn", "pandas", "numpy", "matplotlib", "streamlit", "tensorboard"]

for pkg in required:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "installed")
        print(f"  {pkg:12s} -> OK ({version})")
    except ImportError:
        print(f"  {pkg:12s} -> NOT installed in this kernel")

  tensorflow   -> NOT installed in this kernel
  keras        -> NOT installed in this kernel


  sklearn      -> OK (1.9.0)
  pandas       -> OK (3.0.3)
  numpy        -> OK (2.5.1)


  matplotlib   -> OK (3.11.0)
  streamlit    -> NOT installed in this kernel
  tensorboard  -> OK (2.21.0)


## 7. Summary

- **Goal:** binary classification — predict bank customer churn (`Exited`) from 11 engineered features, using an ANN built with Keras/TensorFlow.
- **Dataset:** `Churn_Modelling.csv`, 10,000 rows, 14 raw columns (3 identifiers + 10 features + 1 target), no missing values, ~20.4% positive class (imbalanced — worth remembering once evaluating the trained model).
- **Pipeline:** feature engineering → build & train ANN (with dropout) → save model (pickle/`.h5`) → Streamlit web app → deploy to Streamlit Cloud.
- **Correction applied:** TensorFlow version corrected from the video's stated `2.1.50` (not a real release) to `2.15.0`.

## 8. What's next

- Feature engineering notebook: encoding `Geography`/`Gender`, standardizing numeric features, train/test split.
- Building and training the ANN with Keras/TensorFlow (loss function, optimizer, dropout, TensorBoard logging).
- Saving the trained model and building the Streamlit app.